## Ocean Cubes Visualization: Local Cluster Zarr Store (Mediterranean Fronts)


In [ ]:
# ==============================================================================
# 0. Start CORS & Local Zarr Proxy (works locally and on EOXHub / JupyterHub)
# ==============================================================================
# - Remote CloudFerro S3 GeoZarrs need CORS headers on HEAD requests.
# - Local cluster Zarr v2/v3 stores (e.g. `bucket/mediterranean_fronts_v3.zarr`)
#   cannot be served via Jupyter's `/files/` route because `ContentsManager`
#   blocks hidden dot-files (`.zmetadata`, `.zgroup`, `.zarray`, `.zattrs`).
# - `cors_proxy.py` handles both: upstream S3 paths (`/...`) and cluster-local
#   files (`/local/...`, with dot-files + HTTP Range/HEAD support).
# - On EOXHub, browser requests reach `127.0.0.1:8010` inside the pod through
#   `jupyter-server-proxy` at `<HUB_ORIGIN><JUPYTERHUB_SERVICE_PREFIX>proxy/8010`.

import http.client
import json
import os
import socket
import subprocess
import sys
import time

PROXY_PORT = 8010
HUB_ORIGIN = os.environ.get("HUB_ORIGIN", "https://jupyterhub.hub-otc-sc.eox.at").rstrip("/")
service_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")

# Serve `/local/...` from the Jupyter Server root / $HOME (matching Jupyter's `/files/` root)
# so notebooks inside subfolders (e.g. `~/ocean_hackathon/4_Visualizations`) can still read `~/bucket/...`.
_home = os.path.abspath(os.path.expanduser("~"))
_cwd = os.path.abspath(os.getcwd())
LOCAL_ROOT = os.path.abspath(
    os.environ.get("JUPYTER_SERVER_ROOT")
    or (_home if os.path.commonpath([_home, _cwd]) == _home else _cwd)
)

if service_prefix:
    PROXY_BASE = f"{HUB_ORIGIN}/{service_prefix.strip('/')}/proxy/{PROXY_PORT}"
else:
    PROXY_BASE = f"http://localhost:{PROXY_PORT}"


def _check_proxy_health(port: int):
    try:
        conn = http.client.HTTPConnection("127.0.0.1", port, timeout=3)
        conn.request("GET", "/healthz")
        resp = conn.getresponse()
        body = resp.read().decode("utf-8", errors="replace")
        conn.close()
        if resp.status == 200:
            data = json.loads(body)
            if "local_root" in data:
                return data
    except Exception:
        pass
    return None


with socket.socket() as s:
    port_in_use = s.connect_ex(("127.0.0.1", PROXY_PORT)) == 0

health = _check_proxy_health(PROXY_PORT) if port_in_use else None

# Restart cors_proxy.py if an older instance lacks `/local/` or was started with a different `--local-root`
if port_in_use and (
    health is None or os.path.abspath(health.get("local_root", "")) != LOCAL_ROOT
):
    if "proxy" in globals() and proxy.poll() is None:
        proxy.terminate()
        proxy.wait(timeout=5)
    else:
        subprocess.run(
            ["pkill", "-f", f"cors_proxy.py.*--port {PROXY_PORT}"],
            check=False,
        )
    for _ in range(20):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", PROXY_PORT)) != 0:
                port_in_use = False
                health = None
                break
        time.sleep(0.1)

if not port_in_use:
    proxy = subprocess.Popen(
        [
            sys.executable,
            "cors_proxy.py",
            "--port",
            str(PROXY_PORT),
            "--local-root",
            LOCAL_ROOT,
        ]
    )
    for _ in range(30):
        health = _check_proxy_health(PROXY_PORT)
        if health is not None:
            break
        time.sleep(0.1)
    print(f"Started CORS & Local Zarr proxy on 127.0.0.1:{PROXY_PORT} (pid {proxy.pid})")
elif health is not None:
    print(f"Proxy already running on 127.0.0.1:{PROXY_PORT}")
else:
    print(
        f"WARNING: Port {PROXY_PORT} is occupied by another process. "
        "Stop that process or change PROXY_PORT above."
    )

# --- Diagnostic Checks ---
if health:
    print(f"  [OK] Internal health check : {health}")
try:
    conn = http.client.HTTPConnection("127.0.0.1", PROXY_PORT, timeout=5)
    conn.request("HEAD", "/EarthCODE/OSCAssets/med_cubes/ocean-med-biophysics.zarr/.zmetadata")
    up_resp = conn.getresponse()
    up_resp.read()
    conn.close()
    print(f"  [OK] Upstream S3 HEAD test : HTTP {up_resp.status} (CORS header: {up_resp.getheader('Access-Control-Allow-Origin')})")
except Exception as exc:
    print(f"  [WARN] Upstream S3 check failed: {exc}")

print(f"\nBrowser Proxy Base URL: {PROXY_BASE}")
if service_prefix:
    print(f"Test EOXHub browser routing by opening: {PROXY_BASE}/healthz")

In [ ]:
# ==============================================================================
# 1. Local Cluster Zarr Store: Mediterranean Fronts (via /local/ proxy route)
# ==============================================================================

import importlib
import json
import os
from pathlib import Path
import ipyeoxelements
import ipyeoxelements.generated

# Reload module if EOxTimecontrol template was already patched in a prior run
if "eox-timecontrol-date" in ipyeoxelements.EOxTimecontrol._esm_template:
    importlib.reload(ipyeoxelements.generated)
    importlib.reload(ipyeoxelements)

from ipyeoxelements import EOxMap, EOxLayercontrol, EOxTimecontrol
from ipywidgets import HBox, Layout, VBox

# Path to the Zarr store on the cluster filesystem (relative to Jupyter root / $HOME or current folder)
local_zarr_rel = "bucket/mediterranean_fronts_v3.zarr"
variable = "fronts"

_home = os.path.abspath(os.path.expanduser("~"))
_cwd = os.path.abspath(os.getcwd())
_local_root = globals().get("LOCAL_ROOT") or os.path.abspath(
    os.environ.get("JUPYTER_SERVER_ROOT")
    or (_home if os.path.commonpath([_home, _cwd]) == _home else _cwd)
)


def _resolve_local_zarr(zarr_path: str, local_root: str) -> tuple[Path, str]:
    """Locate a Zarr store in cwd, ancestor dirs, or Jupyter/LOCAL_ROOT without dereferencing symlinks."""
    root_abs = os.path.abspath(local_root)
    rel_clean = zarr_path.lstrip("/")

    # Walk from cwd up to local_root, then ensure local_root and $HOME are checked
    search_bases = []
    cur = os.path.abspath(os.getcwd())
    while True:
        search_bases.append(cur)
        if cur == root_abs or os.path.dirname(cur) == cur:
            break
        cur = os.path.dirname(cur)
    for extra in (root_abs, os.path.abspath(os.path.expanduser("~"))):
        if extra not in search_bases:
            search_bases.append(extra)

    candidates = []
    for base in search_bases:
        cand = os.path.abspath(
            zarr_path if os.path.isabs(zarr_path) else os.path.join(base, rel_clean)
        )
        if cand not in candidates:
            candidates.append(cand)

    for cand in candidates:
        if os.path.exists(cand):
            if os.path.commonpath([root_abs, cand]) == root_abs:
                url_rel = os.path.relpath(cand, root_abs).replace(os.sep, "/")
                print(f"  [OK] Resolved local store: {cand}")
                return Path(cand), url_rel
            print(
                f"  [WARN] Found '{cand}', but proxy --local-root is '{root_abs}'. "
                "Re-run Cell 0 with LOCAL_ROOT set to a parent folder."
            )
            return Path(cand), rel_clean

    print(f"  [WARN] Local store '{zarr_path}' not found. Checked:")
    for cand in candidates:
        print(f"    - {cand}")
    return Path(candidates[0]), rel_clean


def _detect_geozarr_source_mode(store_path: Path, var_name: str) -> dict:
    """Choose `bands` vs `variable` for ol/source/GeoZarr based on store metadata.

    OpenLayers' GeoZarr uses `bands: [var]` when GeoZarr root attributes
    (`multiscales`, `spatial:bbox`, or `bounds`) are present, and requires
    `variable: var` when those attributes are absent so it infers extent/CRS
    from the `lat`/`lon` coordinate arrays.
    """
    if not store_path.exists():
        return {"bands": [var_name]}

    attrs = {}
    if (store_path / "zarr.json").is_file():
        doc = json.loads((store_path / "zarr.json").read_text())
        attrs = doc.get("attributes", {})
        fmt = "Zarr v3 (zarr.json)"
    elif (store_path / ".zmetadata").is_file():
        doc = json.loads((store_path / ".zmetadata").read_text())
        attrs = doc.get("metadata", {}).get(".zattrs", {})
        fmt = "Zarr v2 consolidated (.zmetadata)"
    elif (store_path / ".zattrs").is_file():
        attrs = json.loads((store_path / ".zattrs").read_text())
        fmt = "Zarr v2 (.zattrs)"
    else:
        fmt = "Zarr (unconsolidated / no root attrs)"

    has_geozarr_attrs = any(k in attrs for k in ("multiscales", "spatial:bbox", "bounds"))
    mode = {"bands": [var_name]} if has_geozarr_attrs else {"variable": var_name}
    print(f"  [OK] Detected {fmt} | GeoZarr attrs: {has_geozarr_attrs} -> using {mode}")
    return mode


resolved_store_path, resolved_url_rel = _resolve_local_zarr(local_zarr_rel, _local_root)

# Verify running proxy's local_root matches _local_root
if "_check_proxy_health" in globals():
    _h = _check_proxy_health(PROXY_PORT)
    if _h and os.path.abspath(_h.get("local_root", "")) != _local_root:
        print(
            f"  [WARN] Running proxy serves '{_h.get('local_root')}' instead of '{_local_root}'. "
            "Re-run Cell 0 first so `/local/` can reach this store!"
        )

# Served through cors_proxy.py's `/local/` route so dot-files (`.zmetadata`, `.zarray`)
# and HTTP Range/HEAD requests work behind EOXHub's jupyter-server-proxy.
data = f"{PROXY_BASE}/local/{resolved_url_rel}"
print("Local GeoZarr URL:", data)

source_band_cfg = _detect_geozarr_source_mode(resolved_store_path, variable)

# ------------------------------------------------------------------------------
# Layer Configurations (EOxElements layer JSON, written as Python dicts)
# ------------------------------------------------------------------------------
layers = [
    {"type": "Tile", "source": {"type": "OSM"}},
    {
        "type": "WebGLTile",
        "properties": {
            "id": "sea-ice-cube",
            "title": "Mediterranean Fronts GeoZarr",
            "layerControlExpand": True,
            "layerControlToolsExpand": True,
            "layerConfig": {
                "type": "style",
                "style": True,
                "autofill": True,
            },
        },
        "source": {
            "type": "GeoZarr",
            "crossOrigin": "anonymous",
            "url": data,
            **source_band_cfg,
        },
        "style": {
            "variables": {
                "min": 0,
                "max": 40,
            },
            "color": [
                "interpolate",
                ["linear"],
                ["band", 1],
                ["var", "min"],
                ["color", 0, 210, 255, ["band", 2]],
                [
                    "+",
                    ["var", "min"],
                    ["*", ["-", ["var", "max"], ["var", "min"]], 0.3333333333333333],
                ],
                ["color", 254, 217, 118, ["band", 2]],
                [
                    "+",
                    ["var", "min"],
                    ["*", ["-", ["var", "max"], ["var", "min"]], 0.6666666666666666],
                ],
                ["color", 253, 141, 60, ["band", 2]],
                ["var", "max"],
                ["color", 189, 0, 38, ["band", 2]],
            ],
        },
    },
]

time_control_file = "./timecontrol.js"
with open(time_control_file, "r") as f:
    time_control_element = f.read()

if "eox-timecontrol-date" not in EOxTimecontrol._esm_template:
    EOxTimecontrol._esm_template = EOxTimecontrol._esm_template.replace(
        "view.el.appendChild(el);", time_control_element
    )

# ------------------------------------------------------------------------------
# Create and Display Map & Widgets
# ------------------------------------------------------------------------------
fronts_map = EOxMap(
    id="fronts-map",
    center=[15, 45],
    zoom=4,
    layout=Layout(width="100%", height="500px"),
)
fronts_map.layers = layers

fronts_time_control = EOxTimecontrol(
    for_="#fronts-map",
    init_date=["first"],
    layout=Layout(width="100%"),
)

fronts_control = EOxLayercontrol(
    for_="#fronts-map",
    tools=["config", "opacity"],
    layout=Layout(flex="1", height="550px", width="300px", min_width="250px"),
)

fronts_map_container = VBox(
    [fronts_map, fronts_time_control],
    layout=Layout(flex="2", display="flex", flex_direction="column", gap="10px"),
)

HBox(
    [fronts_control, fronts_map_container],
    layout=Layout(width="100%", display="flex", gap="16px", align_items="flex-start"),
)

In [ ]:
# Stop the local CORS & Zarr proxy when you are done (the maps above stop loading data).
if "proxy" in globals() and proxy.poll() is None:
    proxy.terminate()
    print("Stopped proxy.")